# Hurst Null Distribution Simulation
Generates null distribution constants for single-scale and multi-scale R/S Hurst estimators
across multiple bit-stream lengths (N=150, 288, 576, 1152).

Output: config values ready to paste into config.js

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Hurst estimators ──────────────────────────────────────────────────────────

def hurst_single(bits):
    """Single-scale R/S — exact match to JavaScript hurstApprox."""
    n = len(bits)
    if n < 20:
        return 0.5
    x = np.where(bits == 1, 1.0, -1.0)
    mean = x.mean()
    d = x - mean
    y = np.cumsum(d)
    R = y.max() - y.min()
    S = np.sqrt((d**2).mean()) or 1.0
    rs = R / S if R / S > 0 else 1.0
    return float(np.clip(np.log(rs) / np.log(n), 0, 1))


def hurst_multi(bits, min_scale=8):
    """Multi-scale R/S via log-log regression across dyadic sub-scales."""
    n = len(bits)
    x = np.where(bits == 1, 1.0, -1.0)

    # Dyadic scales from min_scale up to n//2
    scales = []
    s = min_scale
    while s <= n // 2:
        scales.append(s)
        s *= 2

    if len(scales) < 3:
        return hurst_single(bits)  # not enough scales — fall back

    rs_vals = []
    valid_scales = []
    for scale in scales:
        n_seg = n // scale
        if n_seg == 0:
            continue
        rs_list = []
        for i in range(n_seg):
            seg = x[i*scale:(i+1)*scale]
            d = seg - seg.mean()
            y = np.cumsum(d)
            R = y.max() - y.min()
            S = seg.std(ddof=0) or 1.0
            if R > 0:
                rs_list.append(R / S)
        if rs_list:
            rs_vals.append(np.mean(rs_list))
            valid_scales.append(scale)

    if len(rs_vals) < 3:
        return hurst_single(bits)

    h, _ = np.polyfit(np.log(valid_scales), np.log(rs_vals), 1)
    return float(np.clip(h, 0, 1))

In [ ]:
# ── Simulation ────────────────────────────────────────────────────────────────

N_SIMS   = 10_000
N_VALUES = [150, 288, 576, 1152]
PCTILES  = [10, 25, 50, 75, 90, 95, 99]

rng = np.random.default_rng(42)
results = {}

for n in N_VALUES:
    print(f'Simulating N={n} ({N_SIMS:,} runs)...', end=' ', flush=True)
    single_h = np.empty(N_SIMS)
    multi_h  = np.empty(N_SIMS)
    for i in range(N_SIMS):
        bits = rng.integers(0, 2, n)
        single_h[i] = hurst_single(bits)
        multi_h[i]  = hurst_multi(bits)
    results[n] = {'single': single_h, 'multi': multi_h}
    print(f'single SD={single_h.std():.4f}  multi SD={multi_h.std():.4f}')

print('Done.')

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────

rows = []
for n in N_VALUES:
    for method, arr in results[n].items():
        n_scales = len([s for s in [8, 16, 32, 64, 128, 256, 512] if s <= n // 2])
        row = {
            'N': n,
            'method': method,
            'n_scales': n_scales if method == 'multi' else 1,
            'mean': arr.mean(),
            'sd':   arr.std(),
        }
        for p in PCTILES:
            row[f'p{p}'] = np.percentile(arr, p)
        rows.append(row)

df = pd.DataFrame(rows)
pd.set_option('display.float_format', '{:.5f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
print(df.to_string(index=False))

In [ ]:
# ── Config snippet output ─────────────────────────────────────────────────────

for n in N_VALUES:
    for method in ['single', 'multi']:
        arr = results[n][method]
        print(f'// N={n}, {method}-scale')
        print(f'NULL_HURST_MEAN: {arr.mean():.5f},')
        print(f'NULL_HURST_SD:   {arr.std():.5f},')
        for p in PCTILES:
            print(f'NULL_HURST_P{p:02d}:  {np.percentile(arr, p):.5f},')
        print()

In [ ]:
# ── Distribution plots ────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, len(N_VALUES), figsize=(16, 8), sharey='row')
fig.suptitle('Null Hurst distributions — single vs multi scale', fontsize=14)

for col, n in enumerate(N_VALUES):
    for row, method in enumerate(['single', 'multi']):
        ax = axes[row][col]
        arr = results[n][method]
        color = 'steelblue' if method == 'single' else 'darkorange'
        ax.hist(arr, bins=60, color=color, alpha=0.7, density=True)
        ax.axvline(arr.mean(), color='red', lw=1.5, label=f'mean={arr.mean():.4f}')
        ax.axvline(arr.mean() + arr.std(), color='gray', lw=1, ls='--')
        ax.axvline(arr.mean() - arr.std(), color='gray', lw=1, ls='--')
        ax.set_title(f'N={n} · {method}\nSD={arr.std():.4f}')
        ax.set_xlabel('Hurst')
        if col == 0:
            ax.set_ylabel('Density')
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── SD crossover plot ─────────────────────────────────────────────────────────
# Lower null SD = tighter null = better sensitivity to real effects.
# The crossover point is where multi-scale becomes more informative than single-scale.

sd_single = [results[n]['single'].std() for n in N_VALUES]
sd_multi  = [results[n]['multi'].std()  for n in N_VALUES]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(N_VALUES, sd_single, 'o-', color='steelblue',  label='single-scale')
ax.plot(N_VALUES, sd_multi,  's-', color='darkorange', label='multi-scale')
ax.set_xlabel('N (bits per block)')
ax.set_ylabel('Null SD (lower = more sensitive)')
ax.set_title('Null SD by N and estimator — crossover point')
ax.legend()
ax.grid(True, alpha=0.3)
for i, n in enumerate(N_VALUES):
    ax.annotate(f'{sd_single[i]:.4f}', (n, sd_single[i]), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8, color='steelblue')
    ax.annotate(f'{sd_multi[i]:.4f}',  (n, sd_multi[i]),  textcoords='offset points', xytext=(0, -14), ha='center', fontsize=8, color='darkorange')
plt.tight_layout()
plt.show()